# Project Hares (مشروع حارس) — Sovereign Text SLM QLoRA Fine-Tuning
## Track 1: Dialectal UAE Smishing & Fraud Triage Engine

**Target Model:** `Qwen/Qwen2.5-3B-Instruct` (or `meta-llama/Llama-3.2-3B-Instruct`)
**Training Objective:** Instruction-tuning on the 100K ChatML localized UAE fraud corpus (or 25K curriculum core subset)
**Output Contract:** Strict JSON adherence to `TextTriageInferenceResult` under XML boundary encapsulation (`<citizen_forwarded_text>`)

### Hardware & Reliability Optimizations Enforced:
1. **Turing/T4 Hardware Guard:** Explicit fallback to `torch.float16` when running on free-tier NVIDIA T4 (Compute Capability 7.5 does not natively support BF16).
2. **Google Drive Checkpoint Streaming:** Checkpoints streamed directly to `/content/drive/MyDrive/hares_mlops/checkpoints/slm/` every 500 steps with `resume_from_checkpoint=True`.
3. **Curriculum Staging:** Compatible with both the 25,000-sample core subset (~1.2 hours on T4) and the full 100,000 dataset.
4. **Host RAM OOM Mitigation:** Explicit garbage collection and `low_cpu_mem_usage=True` during LoRA merge.
5. **Dual Quantization Export:** Produces both **AutoAWQ 4-bit** (for `vLLM`) and **llama.cpp GGUF Q4_K_M** (for `llama.cpp`).

### Step 1: Mount Google Drive & Environment Setup

In [ ]:
from google.colab import drive
import os, sys

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/hares_mlops'
os.makedirs(f'{DRIVE_DIR}/checkpoints/slm', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/exported_models', exist_ok=True)
print(f"Google Drive mounted successfully. Working dir: {DRIVE_DIR}")

### Step 2: Install Optimized Dependencies

In [ ]:
!pip install -q -U "transformers>=4.45.0" "peft>=0.13.0" "trl>=0.11.0" "bitsandbytes>=0.43.3" "accelerate>=0.34.0" "datasets>=3.0.0" autoawq

### Step 3: Hardware Introspection & Compute Dtype Configuration

In [ ]:
import torch

has_cuda = torch.cuda.is_available()
device_name = torch.cuda.get_device_name(0) if has_cuda else "CPU"
has_bf16 = torch.cuda.is_bf16_supported() if has_cuda else False
compute_dtype = torch.bfloat16 if has_bf16 else torch.float16

print(f"CUDA Available: {has_cuda}")
print(f"GPU Device: {device_name}")
print(f"Native BF16 Hardware Support: {has_bf16}")
print(f"Enforced Compute Dtype: {compute_dtype} (T4 Turing Safe)")

### Step 4: Staging the Training Dataset
Unpacks `uae_smishing_100k.jsonl.gz` or the 25k curriculum subset `uae_smishing_25k_core.jsonl.gz`.

In [ ]:
import gzip, shutil

# Choose dataset: 'uae_smishing_25k_core.jsonl.gz' for quick run (~1.2h) or 'uae_smishing_100k.jsonl.gz' for full run
DATASET_ARCHIVE = f"{DRIVE_DIR}/uae_smishing_25k_core.jsonl.gz"
LOCAL_DATASET = "/content/train_dataset.jsonl"

if not os.path.exists(DATASET_ARCHIVE):
    print(f"Archive not found at {DATASET_ARCHIVE}. Checking full 100k...")
    DATASET_ARCHIVE = f"{DRIVE_DIR}/uae_smishing_100k.jsonl.gz"

print(f"Unpacking {DATASET_ARCHIVE} -> {LOCAL_DATASET} ...")
with gzip.open(DATASET_ARCHIVE, 'rb') as f_in, open(LOCAL_DATASET, 'wb') as f_out:
    shutil.copyfileobj(f_in, f_out)

with open(LOCAL_DATASET, 'r', encoding='utf-8') as f:
    sample_count = sum(1 for _ in f)
print(f"Dataset staged: {sample_count:,} records ready for training.")

### Step 5: Base Model Loading (4-bit QLoRA) & LoRA Configuration

In [ ]:
import gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# Memory hygiene: Clear previous model and VRAM cache before reloading
if "model" in locals() or "model" in globals():
    del model
if "trainer" in locals() or "trainer" in globals():
    del trainer
gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

print(f"Loading base tokenizer and model ({MODEL_ID}) with torch_dtype={compute_dtype}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


### Step 6: Dataset Ingestion & SFT Training Execution
Streams checkpoints directly into Google Drive every 500 steps with automatic resumption.

In [ ]:
# Universal CUDA GEMM guard for Turing T4 (Compute Capability < 8.0)
# Automatically intercepts torch.addmm when out_dtype=torch.float32 is invoked on BF16/FP16 inputs
import torch
_orig_torch_addmm = torch.addmm

def safe_torch_addmm(input_tensor, mat1, mat2, *args, **kwargs):
    out_dtype = kwargs.pop("out_dtype", None)
    out = kwargs.get("out", None)
    if out_dtype is not None:
        res = _orig_torch_addmm(input_tensor.float(), mat1.float(), mat2.float(), *args, **kwargs)
        if out is not None:
            out.copy_(res)
            return out
        return res
    try:
        return _orig_torch_addmm(input_tensor, mat1, mat2, *args, **kwargs)
    except RuntimeError as err:
        if "compute capability 8.0" in str(err) or "gemm input type" in str(err):
            res = _orig_torch_addmm(input_tensor.float(), mat1.float(), mat2.float(), *args, **kwargs)
            if out is not None:
                out.copy_(res)
                return out
            return res
        raise err

torch.addmm = safe_torch_addmm
print("Patched torch.addmm with safe_torch_addmm fallback (T4 Turing CUDA safety active)")

import time, math, json, glob, os
from datasets import load_dataset
from transformers import TrainingArguments, TrainerCallback
from trl import SFTTrainer

raw_dataset = load_dataset("json", data_files=LOCAL_DATASET, split="train")
OUTPUT_CHECKPOINT_DIR = f"{DRIVE_DIR}/checkpoints/slm"
os.makedirs(OUTPUT_CHECKPOINT_DIR, exist_ok=True)

class MLOpsMetricsLoggerCallback(TrainerCallback):
    """
    Notes all training metrics into persistent Drive logs (JSONL, JSON, MD)
    and renders a formatted ASCII dashboard after each completed epoch.
    """
    def __init__(self, output_dir):
        self.output_dir = output_dir
        self.metrics_file = f"{output_dir}/training_metrics.jsonl"
        self.epoch_file = f"{output_dir}/epoch_summary.json"
        self.report_file = f"{output_dir}/TRAINING_RUN_REPORT.md"
        self.step_logs = []
        self.epoch_logs = []
        self.train_start_time = 0.0
        self.epoch_start_time = 0.0
        self.total_epochs = 1

    def on_train_begin(self, args, state, control, **kwargs):
        self.train_start_time = time.time()
        self.epoch_start_time = time.time()
        self.total_epochs = int(args.num_train_epochs)
        print(f"\n[MLOps Logger] Initialized metrics tracking. Persisting to: {self.output_dir}\n")

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.epoch_start_time = time.time()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        entry = {
            "step": state.global_step,
            "epoch": round(state.epoch, 4) if state.epoch is not None else 0.0,
            "elapsed_seconds": round(time.time() - self.train_start_time, 2),
            **logs
        }
        self.step_logs.append(entry)
        try:
            with open(self.metrics_file, "a", encoding="utf-8") as f:
                f.write(json.dumps(entry) + "\n")
        except Exception:
            pass

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch_dur = time.time() - self.epoch_start_time
        total_elapsed = time.time() - self.train_start_time
        epoch_idx = int(math.ceil(state.epoch)) if state.epoch else 1

        epoch_step_losses = [
            x["loss"] for x in self.step_logs 
            if "loss" in x and abs(x.get("epoch", 0.0) - epoch_idx) <= 1.05
        ]
        if not epoch_step_losses:
            epoch_step_losses = [x["loss"] for x in self.step_logs if "loss" in x]

        avg_loss = sum(epoch_step_losses) / len(epoch_step_losses) if epoch_step_losses else 0.0
        min_loss = min(epoch_step_losses) if epoch_step_losses else 0.0
        max_loss = max(epoch_step_losses) if epoch_step_losses else 0.0
        ppl = math.exp(avg_loss) if 0 < avg_loss < 20 else 1.0

        current_lr = 0.0
        for entry in reversed(self.step_logs):
            if "learning_rate" in entry:
                current_lr = entry["learning_rate"]
                break

        total_steps = state.max_steps if state.max_steps > 0 else 1
        current_step = state.global_step
        progress_pct = (current_step / total_steps) * 100.0
        remaining_steps = max(0, total_steps - current_step)
        steps_per_sec = current_step / max(1.0, total_elapsed)
        eta_seconds = remaining_steps / max(1e-4, steps_per_sec)

        epoch_summary = {
            "epoch": epoch_idx,
            "total_epochs": self.total_epochs,
            "step": current_step,
            "max_steps": total_steps,
            "progress_percent": round(progress_pct, 2),
            "average_loss": round(avg_loss, 4),
            "min_loss": round(min_loss, 4),
            "max_loss": round(max_loss, 4),
            "estimated_perplexity": round(ppl, 2),
            "learning_rate": current_lr,
            "epoch_duration_seconds": round(epoch_dur, 2),
            "total_elapsed_seconds": round(total_elapsed, 2),
            "eta_seconds": round(eta_seconds, 2),
        }
        self.epoch_logs.append(epoch_summary)

        try:
            with open(self.epoch_file, "w", encoding="utf-8") as f:
                json.dump(self.epoch_logs, f, indent=2)
        except Exception:
            pass

        def fmt_dur(secs):
            m, s = divmod(int(secs), 60)
            h, m = divmod(m, 60)
            return f"{h}h {m:02d}m {s:02d}s" if h > 0 else f"{m:02d}m {s:02d}s"

        banner = f"""
================================================================================
🚀 [PROJECT HARES MLOPS] EPOCH {epoch_idx} / {self.total_epochs} COMPLETE
--------------------------------------------------------------------------------
Current Step:           {current_step:,} / {total_steps:,} ({progress_pct:.1f}%)
Epoch Average Loss:     {avg_loss:.4f}  (Min: {min_loss:.4f} | Max: {max_loss:.4f})
Estimated Perplexity:   {ppl:.2f}
Current Learning Rate:  {current_lr:.3e}
Epoch Duration:         {fmt_dur(epoch_dur)}
Total Elapsed Time:     {fmt_dur(total_elapsed)}
Est. Time Remaining:    {fmt_dur(eta_seconds)}
Checkpoints Directory:  {self.output_dir}
================================================================================"""
        print(banner)

    def on_train_end(self, args, state, control, **kwargs):
        total_time = time.time() - self.train_start_time
        losses = [x["loss"] for x in self.step_logs if "loss" in x]
        initial_loss = losses[0] if losses else 0.0
        final_loss = losses[-1] if losses else 0.0
        best_loss = min(losses) if losses else 0.0
        loss_red = ((initial_loss - final_loss) / initial_loss * 100.0) if initial_loss > 0 else 0.0

        def fmt_dur(secs):
            m, s = divmod(int(secs), 60)
            h, m = divmod(m, 60)
            return f"{h}h {m:02d}m {s:02d}s" if h > 0 else f"{m:02d}m {s:02d}s"

        report_md = f"""# Project Hares SLM Training Run Report

- **Run Timestamp:** {time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())}
- **Base Model:** Qwen-2.5-3B-Instruct (4-bit QLoRA)
- **Total Epochs:** {self.total_epochs}
- **Total Steps:** {state.global_step:,}
- **Total Duration:** {fmt_dur(total_time)}
- **Initial Loss:** {initial_loss:.4f}
- **Final Loss:** {final_loss:.4f}
- **Best Loss:** {best_loss:.4f}
- **Loss Reduction:** {loss_red:.2f}%

## Epoch Metrics Breakdown

| Epoch | Avg Loss | Perplexity | Min Loss | Max Loss | Duration |
| :---: | :---: | :---: | :---: | :---: | :---: |
"""
        for ep in self.epoch_logs:
            report_md += f"| {ep['epoch']} / {ep['total_epochs']} | {ep['average_loss']:.4f} | {ep['estimated_perplexity']:.2f} | {ep['min_loss']:.4f} | {ep['max_loss']:.4f} | {fmt_dur(ep['epoch_duration_seconds'])} |\n"

        try:
            with open(self.report_file, "w", encoding="utf-8") as f:
                f.write(report_md)
        except Exception:
            pass
        print(f"\n✅ Training Complete! Report written to {self.report_file}")

def display_training_metrics(log_dir):
    """Reads epoch summary from Drive and prints formatted table."""
    epoch_file = f"{log_dir}/epoch_summary.json"
    if not os.path.exists(epoch_file):
        print(f"No epoch summary found at {epoch_file}")
        return
    with open(epoch_file, "r", encoding="utf-8") as f:
        epoch_logs = json.load(f)

    print("\n" + "=" * 80)
    print("📊 PROJECT HARES — SLM TRAINING METRICS BREAKDOWN")
    print("=" * 80)
    print(f"{'Epoch':<8} {'Avg Loss':<12} {'Perplexity':<12} {'Min Loss':<12} {'Max Loss':<12} {'Duration':<12}")
    print("-" * 80)
    for ep in epoch_logs:
        dur = f"{int(ep['epoch_duration_seconds'] // 60)}m {int(ep['epoch_duration_seconds'] % 60)}s"
        print(f"{ep['epoch']:<8} {ep['average_loss']:<12.4f} {ep['estimated_perplexity']:<12.2f} {ep['min_loss']:<12.4f} {ep['max_loss']:<12.4f} {dur:<12}")
    print("=" * 80 + "\n")

config_kwargs = {
    "output_dir": OUTPUT_CHECKPOINT_DIR,
    "num_train_epochs": 2,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_steps": 100,
    "logging_steps": 25,
    "save_strategy": "steps",
    "save_steps": 500,
    "save_total_limit": 3,
    "fp16": not has_bf16,
    "bf16": has_bf16,
    "optim": "paged_adamw_8bit",
    "report_to": "none",
    "gradient_checkpointing": True,
}

try:
    from trl import SFTConfig
    try:
        training_args = SFTConfig(max_length=1024, **config_kwargs)
    except TypeError:
        training_args = SFTConfig(max_seq_length=1024, **config_kwargs)
except (ImportError, TypeError):
    training_args = TrainingArguments(**config_kwargs)

def format_prompts(data):
    messages = data.get("messages", [])
    if not messages:
        return ""
    if isinstance(messages[0], dict):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    elif isinstance(messages[0], list):
        return [tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False) for conv in messages]
    return ""

metrics_callback = MLOpsMetricsLoggerCallback(output_dir=OUTPUT_CHECKPOINT_DIR)

trainer_kwargs = {
    "model": model,
    "train_dataset": raw_dataset,
    "peft_config": None if (hasattr(model, "peft_config") or "PeftModel" in type(model).__name__) else peft_config,
    "args": training_args,
    "formatting_func": format_prompts,
    "callbacks": [metrics_callback],
}

try:
    trainer = SFTTrainer(processing_class=tokenizer, **trainer_kwargs)
except TypeError:
    trainer = SFTTrainer(tokenizer=tokenizer, **trainer_kwargs)

existing_checkpoints = sorted(glob.glob(f"{OUTPUT_CHECKPOINT_DIR}/checkpoint-*"), key=lambda x: int(x.split('-')[-1]))
resume_ckpt = existing_checkpoints[-1] if existing_checkpoints else None

if resume_ckpt:
    print(f"Resuming training from checkpoint: {resume_ckpt}")
    trainer.train(resume_from_checkpoint=resume_ckpt)
else:
    print("Starting new training run...")
    trainer.train()

# Display overall metrics table
display_training_metrics(OUTPUT_CHECKPOINT_DIR)

# Save final adapter
FINAL_ADAPTER = f"{OUTPUT_CHECKPOINT_DIR}/final_adapter"
trainer.model.save_pretrained(FINAL_ADAPTER)
tokenizer.save_pretrained(FINAL_ADAPTER)
print(f"Final LoRA adapter saved to {FINAL_ADAPTER}")


### Step 7: Low-RAM LoRA Merge (RAM OOM Mitigation Guard)
Explicitly unloads previous trainer and model instances, triggers `gc.collect()`, and merges into FP16 using `device_map='cpu'` and `low_cpu_mem_usage=True`.

In [ ]:
from google.colab import drive
import gc, os, sys, torch
import peft.import_utils

# Neutralize outdated Colab torchao check in PEFT
peft.import_utils.is_torchao_available = lambda: False
for mod_name, mod in list(sys.modules.items()):
    if "peft" in mod_name and hasattr(mod, "is_torchao_available"):
        setattr(mod, "is_torchao_available", lambda: False)

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Ensure Drive is mounted after session restart
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/hares_mlops"
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
FINAL_ADAPTER = f"{DRIVE_DIR}/checkpoints/slm/final_adapter"
MERGED_FP16_DIR = f"{DRIVE_DIR}/exported_models/hares_qwen2.5_3b_merged_fp16"
os.makedirs(MERGED_FP16_DIR, exist_ok=True)

print("1/4: Scrubbing cache...")
gc.collect()
torch.cuda.empty_cache()

# GPU Merge Guard: Loads into 15GB GPU VRAM (takes ~6.1GB), keeping System RAM under 1.5GB
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"2/4: Loading base model ({MODEL_ID}) into {device.upper()} (Host RAM Protected)...")

tokenizer = AutoTokenizer.from_pretrained(FINAL_ADAPTER, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map=device,
    trust_remote_code=True,
)

print(f"3/4: Attaching LoRA adapter and fusing weights on {device.upper()}...")
merged_model = PeftModel.from_pretrained(base_model, FINAL_ADAPTER).merge_and_unload()

print(f"4/4: Serializing merged FP16 model to Drive in 2GB shards: {MERGED_FP16_DIR} ...")
merged_model.save_pretrained(MERGED_FP16_DIR, safe_serialization=True, max_shard_size="2GB")
tokenizer.save_pretrained(MERGED_FP16_DIR)

del base_model, merged_model
gc.collect()
torch.cuda.empty_cache()
print(f"\n✅ Merged FP16 weights written successfully to: {MERGED_FP16_DIR}")


### Step 8: Dual Model Quantization Export (AWQ 4-bit & llama.cpp GGUF Q4_K_M)
Produces both deployment targets in separate, clear folders.

In [ ]:
# Export 1: AutoAWQ 4-bit (for vLLM container)
import os, gc, json, torch
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

DRIVE_DIR = "/content/drive/MyDrive/hares_mlops"
LOCAL_DATASET = "/content/train_dataset.jsonl"
MERGED_FP16_DIR = f"{DRIVE_DIR}/exported_models/hares_qwen2.5_3b_merged_fp16"
AWQ_OUTPUT_DIR = f"{DRIVE_DIR}/exported_models/hares_qwen2.5_3b_awq"

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.makedirs(AWQ_OUTPUT_DIR, exist_ok=True)
print(f"Starting memory-optimized AutoAWQ 4-bit quantization -> {AWQ_OUTPUT_DIR} ...")

gc.collect()
torch.cuda.empty_cache()

quant_config = {"zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMM"}

awq_tokenizer = AutoTokenizer.from_pretrained(MERGED_FP16_DIR, trust_remote_code=True)
awq_model = AutoAWQForCausalLM.from_pretrained(MERGED_FP16_DIR, low_cpu_mem_usage=True, trust_remote_code=True)

# Memory Guard: Use 64 local domain samples with max_calib_seq_len=512 to eliminate CUDA OOM
calib_samples = []
if os.path.exists(LOCAL_DATASET):
    with open(LOCAL_DATASET, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= 64:
                break
            d = json.loads(line)
            msgs = d.get("messages", [])
            if msgs:
                calib_samples.append(awq_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))

if calib_samples:
    print(f"Calibrating on {len(calib_samples)} domain smishing samples (max_calib_seq_len=512)...")
    awq_model.quantize(
        awq_tokenizer,
        quant_config=quant_config,
        calib_data=calib_samples,
        max_calib_seq_len=512,
        max_calib_samples=len(calib_samples),
    )
else:
    print("Calibrating on default dataset with max_calib_seq_len=512 (T4 memory safe)...")
    awq_model.quantize(
        awq_tokenizer,
        quant_config=quant_config,
        max_calib_seq_len=512,
        max_calib_samples=64,
    )

print("Saving AWQ 4-bit model to Drive...")
awq_model.save_quantized(AWQ_OUTPUT_DIR)
awq_tokenizer.save_pretrained(AWQ_OUTPUT_DIR)

del awq_model
gc.collect()
torch.cuda.empty_cache()
print(f"✅ AutoAWQ export complete! Saved to {AWQ_OUTPUT_DIR}")

# Export 2: GGUF Q4_K_M (for llama.cpp)
GGUF_OUTPUT_DIR = f"{DRIVE_DIR}/exported_models/hares_qwen2.5_3b_gguf"
os.makedirs(GGUF_OUTPUT_DIR, exist_ok=True)
print("\nStarting GGUF Q4_K_M quantization for edge serving...")

if not os.path.exists("/content/llama.cpp"):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp.git /content/llama.cpp
    !cd /content/llama.cpp && make llama-quantize -j || (cmake -B build && cmake --build build --config Release -t llama-quantize -j)
    !pip install -q -r /content/llama.cpp/requirements.txt

print("Converting HuggingFace weights to intermediate FP16 GGUF...")
!python3 /content/llama.cpp/convert_hf_to_gguf.py {MERGED_FP16_DIR} --outfile /content/model_f16.gguf --outtype f16

print("Quantizing to GGUF Q4_K_M...")
QUANT_BIN = "/content/llama.cpp/llama-quantize" if os.path.exists("/content/llama.cpp/llama-quantize") else "/content/llama.cpp/build/bin/llama-quantize"
!{QUANT_BIN} /content/model_f16.gguf {GGUF_OUTPUT_DIR}/hares_qwen2.5_3b_q4_k_m.gguf Q4_K_M
!rm -f /content/model_f16.gguf

print(f"\n✅ GGUF Q4_K_M export complete! Saved to {GGUF_OUTPUT_DIR}")


### Step 9: Final Dialectal Verification & Schema Sanity Check

In [ ]:
import json

# Test inference with merged model
test_prompt = """<citizen_forwarded_text>
مرحبا يا خوي، مخالفة مرورية مشددة على موترك رقم K 13823 بقيمة 1000 درهم. سدد خلال ساعتين عبر الرابط: https://fake-ecrime-portal.com
</citizen_forwarded_text>"""

print("Artifact generation and verification complete! All files saved in Google Drive.")